# DTCR-U-Net: Kaggle Training & Evaluation Pipeline
This notebook trains the **Dual-Task Consistency Regularized U-Net (DTCR-U-Net)** on Kaggle GPU.

### Step 1: Ensure Kaggle GPU is Enabled
Go to **Settings** (right-hand panel) -> **Accelerator** -> **GPU T4 x2** or **P100**.

In [ ]:
# Clone repo or pull latest updates
!git clone https://github.com/shadyOg/fyp-dtcr-unet.git || (cd fyp-dtcr-unet && git pull origin main)
%cd fyp-dtcr-unet/dtcr-unet
!pip install -q -r requirements.txt

### Step 2: Run Data Preprocessing
Extracts 2D slices, performs HU windowing, CLAHE, Level Set Map computation ($T(y)$), and creates patient-level train/val/test splits.

In [ ]:
import os

# Preprocessing output folder
PROCESSED_OUT = '/kaggle/working/data/processed'

!python data/preprocessing.py \
    --mosmed /kaggle/input \
    --out {PROCESSED_OUT} \
    --size 256 \
    --context 1 \
    --labeled-ratio 0.4

### Step 3: Train DTCR-U-Net (Dual-Task Semi-Supervised with AMP)
Runs 80 epochs with AdamW, Cosine Annealing, AMP mixed precision (FP16), and dynamic consistency weight $\lambda_d(t)$.

In [ ]:
!python train.py \
    --data {PROCESSED_OUT} \
    --epochs 80 \
    --batch-size 8 \
    --grad-accum 2 \
    --lr 0.001 \
    --labeled-ratio 0.4 \
    --checkpoints /kaggle/working/checkpoints \
    --outputs /kaggle/working/outputs

### Step 4: Evaluate Model on Test Set
Computes Dice, Sensitivity (SE), Specificity (SP), Accuracy, F1, and Hausdorff Distance (HD) on the test set, and saves visual segmentation comparisons.

In [ ]:
!python evaluate.py \
    --checkpoint /kaggle/working/checkpoints/best_model.pth \
    --data {PROCESSED_OUT} \
    --out /kaggle/working/outputs/evaluation \
    --batch-size 8